# Web Scraping Opinión - El Español
Scraper de las 10 últimas noticias de la sección de Opinión (para sistema diario automatizado).

In [1]:

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from datetime import datetime


In [19]:

SECCIONES_BASE = [
    "https://www.elespanol.com/opinion/",
]

MAX_PAGINAS = 4
MAX_ARTICULOS = 13

HEADERS = {"User-Agent": "Mozilla/5.0"}


In [20]:

def get_soup(url):
    r = requests.get(url, headers=HEADERS, timeout=10)
    r.raise_for_status()
    return BeautifulSoup(r.text, "html.parser")

def es_articulo(url):
    return "/opinion/" in url and url.count("/") > 4


In [21]:

urls = []
urls_set = set()

for seccion in SECCIONES_BASE:
    print(f"Sección: {seccion}")

    for pagina in range(1, MAX_PAGINAS + 1):

        if len(urls) >= MAX_ARTICULOS:
            break

        if pagina == 1:
            url = seccion
        else:
            url = f"{seccion}{pagina}/"

        print(f"  Página {pagina}")

        try:
            soup = get_soup(url)
            enlaces = soup.select("a[href]")

            nuevos = 0

            for a in enlaces:
                href = a["href"]

                if href.startswith("https://") and es_articulo(href):
                    if href not in urls_set:
                        urls_set.add(href)
                        urls.append(href)
                        nuevos += 1

                if len(urls) >= MAX_ARTICULOS:
                    break

            print(f"    +{nuevos} nuevos (total {len(urls)})")

        except Exception as e:
            print("Error:", e)

        time.sleep(0.5)

    if len(urls) >= MAX_ARTICULOS:
        break

print(f"Total URLs: {len(urls)}")


Sección: https://www.elespanol.com/opinion/
  Página 1
    +13 nuevos (total 13)
Total URLs: 13


In [22]:

def scrape_article(url):
    soup = get_soup(url)

    title = soup.find("h1")
    title = title.get_text(strip=True) if title else ""

    paragraphs = soup.select("article p")
    text = " ".join(p.get_text(strip=True) for p in paragraphs)

    return {
        "periodico": "el español",
        "fecha_scrapeo": datetime.now().strftime("%Y-%m-%d"),
        "titulo": title.lower(),
        "texto": text.lower(),
        "url": url.lower()
    }

rows = []

for url in urls:

    try:
        art = scrape_article(url)

        if art["titulo"] and art["texto"]:
            rows.append(art)

        if len(rows) >= MAX_ARTICULOS:
            break

        time.sleep(0.5)

    except Exception as e:
        print("Error:", e)

print("Artículos scrapeados:", len(rows))


Artículos scrapeados: 11


In [23]:

df = pd.DataFrame(rows)
fecha = datetime.now().strftime("%Y-%m-%d")
df.to_csv(f"elespanol_opinion_{fecha}.csv", index=False, encoding="utf-8-sig")
df.head(10)


,periodico,fecha_scrapeo,titulo,texto,url
0,el español,2026-05-13,carta del director,,https://www.elespanol.com/opinion/carta-del-di...
1,el español,2026-05-13,editoriales,"con sus constantes traspiés, maría jesús mont...",https://www.elespanol.com/opinion/editoriales/
2,el español,2026-05-13,columnas,juanma moreno transmite la convicción de que ...,https://www.elespanol.com/opinion/columnas/
3,el español,2026-05-13,tribunas,la guerra podría terminar con una victoria ru...,https://www.elespanol.com/opinion/tribunas/
4,el español,2026-05-13,sánchez ha transformado al pp en el verdadero ...,"juanma moreno y feijóo, el pasado noviembre. s...",https://www.elespanol.com/opinion/editoriales/...
5,el español,2026-05-13,"misión incumplida, señora ministra: ¿quién asu...",el director general de la organización mundial...,https://www.elespanol.com/opinion/editoriales/...
6,el español,2026-05-13,el 'reality' del hantavirus: sánchez se llevó ...,"marlaska, mónica garcía y ángel víctor torres,...",https://www.elespanol.com/opinion/editoriales/...
7,el español,2026-05-13,"""¡es calígula! ¡el presidente es calígula!""",apenas habían pasado cincuenta horas de la com...,https://www.elespanol.com/opinion/carta-del-di...
8,el español,2026-05-13,españa es un capítulo de benny hill,crucero mv hondius.reuters como me decís que p...,https://www.elespanol.com/opinion/columnas/202...
9,el español,2026-05-13,españa es un capítulo de benny hill,crucero mv hondius.reuters como me decís que p...,https://www.elespanol.com/opinion/columnas/202...
